# AED Placement Optimization

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Polygon

## 1. Importing Data

In [ ]:
# setting parameters for Papermill
input_data_path1 = 'Results/preprocessed_data_with_distances.csv'
input_data_path2 = 'Results/rta_data.csv'
url2 = 'https://raw.githubusercontent.com/JeroenGuillierme/Project-MDA/main/Data/'
output_gdf_interventions_path = 'Results/gdf_interventions_with_counts.csv'
output_aed_path = 'Results/new_aed_locations.csv'

In [ ]:
interventions_data = pd.read_csv(input_data_path2)
aed_data =pd.read_csv(input_data_path1)

# Load Belgium shapefile
belgium_boundary = gpd.read_file(f'{url2}Belgi%C3%AB.json')
# Load Belgium with regions shapefile
belgium_with_provinces_boundary = gpd.read_file(f'{url2}BELGIUM_-_Provinces.geojson')

pd.set_option('display.max_columns', None)

## 2. Functions

Here a custom function is imported from the Functions.py script, which will be used further in this Notebook.

In [ ]:
from Functions import calculate_cell_size

## 3. Data Preparation

Create two different datasets which only contain the intervention and AED locations respectively.

In [ ]:
interventions_locations = interventions_data

aed_locations = aed_data[aed_data['AED'] == 1]
# Reset indices
aed_locations.reset_index(drop=True, inplace=True)
print(f'There are currently {len(aed_locations)} AEDs in Belgium')


In [ ]:
# Check for missing values
print('Missing values per variable: \n', aed_locations.isnull().sum())
print('Length dataset: ', len(aed_locations))

### 3.1 Only keep the one observation per Mission ID with the shortest response time

Only one observation for each Mission ID should be kept, namely the observation of the vector type with the shortest response time. Otherwise, some locations will be counted double which can bias the results of the analysis. When, for the same Mission ID, the different vector types have the same response time, the observation of the Ambulance will be prioritized. The observation with the shortest response time is chosen because it is the first emergency service present on the incident location that matters.

This ensures that the same intervention, with the same Mission ID but different vector type, will not be counted multiple times later on.

In [ ]:
# Sort by Response Time (T3-T0) first and then by vector type to prioritize Ambulance if the times are equal
# Remove duplicate Mission IDs, keeping only the observation with the shortest response time
df = interventions_locations.sort_values(by=['T3-T0', 'Vector type'], ascending=[True, True]).drop_duplicates(
    subset='Mission ID', keep='first')

# Reset indices
df.reset_index(drop=True, inplace= True)

print('There are', len(df), 'unique Mission IDs')
df.head(5)

### 3.2 Remove duplicated values from the dataset

Duplicate AED and/or intervention locations in datasets removed to avoid counting the amount of AEDs and interventions double.

**For the AED Data**

In [ ]:
print('------------- Before removing duplicates -------------')
# Verify the imputation
len1=len(aed_locations)
print('Total length of AED locations dataset: ', len1)

aed_locations_without_dups = aed_locations.drop_duplicates()

print('------------- After removing duplicates -------------')
# Verify the imputation
len2=len(aed_locations_without_dups)
print('Total length of unique AED locations dataset: ', len2)
print('In total: ', len1-len2, ' duplicates removed.')

# Visual check if the rows are in fact exactly the same
aed_locations[aed_locations.duplicated(keep=False)==True].sort_values(by=['Latitude', 'Longitude']).head(6) 

**For the Interventions Data**

In [ ]:
print('------------- Before removing duplicates -------------')
# Verify the imputation
len1=len(df)
print('Total length of interventions dataset: ', len1)

df_without_dups = df.drop_duplicates()

print('------------- After removing duplicates -------------')
# Verify the imputation
len2=len(df_without_dups)
print('Total length of unique interventions dataset: ', len2)
print('In total: ', len1-len2, ' duplicates removed.')

## 4. Heatmap of Incident Desnity

### 4.1 Create Kernel Density Estimate (KDE) plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.set(style='whitegrid')
kde = sns.kdeplot(data=df, x='Longitude', y='Latitude', cmap='Reds',
                  fill=True, ax=ax, cbar=True)
# Plot Belgian Boundary
belgium_boundary.plot(ax=ax, facecolor='none', edgecolor='black')
ax.set_title('Heatmap of Incident Density')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

### 4.2 Create a GeaDataframe from the interventions data

In [ ]:
# Create a GeoDataFrame from interventions data
gdf_interventions = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.Longitude, df.Latitude)
)
print(len(gdf_interventions))

# Filter out invalid geometries
gdf_interventions = gdf_interventions[gdf_interventions['geometry'].is_valid]
print(len(gdf_interventions)) # No invalid geometries found

### 4.3 Adding Incident Density to Dataset

#### 4.3.1 Define size of the grid area in which Belgium will be divided

In [ ]:
# Define size grid area in which we will divide Beligum
grid_area_km2 = 3  # Desired grid area in km²

# Calculate mean latitude from interventions locations
mean_latitude_int = gdf_interventions.geometry.centroid.y.mean()
print(f'The mean latitude of the interventions is: {mean_latitude_int}')

# Calculate mean latitude from Belgium
mean_latitude_bel = belgium_boundary.geometry.centroid.y.mean()
print(f'The mean latitude of Belgium is: {mean_latitude_bel}')

# Calculate cell size in decimal degrees
cell_size_degrees = calculate_cell_size(grid_area_km2, mean_latitude_bel)
print(f'The cell size in degrees is: {cell_size_degrees}')

#### 4.3.2 Create the grid over the study area, namely Belgium

In [ ]:
# Create a grid over the study area: Belgium
xmin, ymin, xmax, ymax = belgium_boundary.total_bounds
cell_size = 0.025  # Adjust the cell size as needed
grid_cells = []
for x0 in np.arange(xmin, xmax + cell_size_degrees, cell_size_degrees):
    for y0 in np.arange(ymin, ymax + cell_size_degrees, cell_size_degrees):
        x1 = x0 - cell_size_degrees
        y1 = y0 + cell_size_degrees
        grid_cells.append(Polygon([(x0, y0), (x1, y0), (x1, y1), (x0, y1)]))

grid = gpd.GeoDataFrame(grid_cells, columns=['geometry'])
print(len(grid))

# Filter out invalid geometries
grid = grid[grid['geometry'].is_valid]
print(len(grid)) # Zero invalid geometries filtered out

#### 4.3.3 Count number of incidents per grid and add to dataset

In [ ]:
# Spatial join the grid with the intervention location points
joined_interventions = gpd.sjoin(gdf_interventions, grid, how='left')

# Count incidents in each grid cell
incident_counts = joined_interventions.groupby('index_right').size()

# Map incidents counts to the grid
grid['incident_count'] = 0 # Initialize the incident counts per grid as zero
grid.loc[incident_counts.index, 'incident_count'] = incident_counts.values
# print(grid['incident_count'].sort_values(ascending=True).value_counts())

# Perform a spatial join to add the incident_count to the interventions data
gdf_interventions_with_incident_count = gpd.sjoin(gdf_interventions, grid[['geometry', 'incident_count']], how='left')

# Drop the index right column
gdf_interventions_with_incident_count = gdf_interventions_with_incident_count.drop(columns='index_right')

#### 4.3.4 Visualization of self-counted Incident Density

In [ ]:
# Plot the incident density
fig, ax = plt.subplots(figsize=(12, 10))
grid.plot(column='incident_count', ax=ax, cmap='OrRd', alpha=0.5, legend=True,
         legend_kwds={"label": "Number of Interventions per Grid", "orientation": "horizontal"})
belgium_boundary.plot(ax=ax, facecolor='none', edgecolor='black')
ax.set_title('Incident Density Grid')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()

## 5. Heatmap of AED Density

### 5.1 Create Kernel Density Estimate (KDE) plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

kde = sns.kdeplot(data=aed_locations_without_dups, x='Longitude', y='Latitude', cmap='Reds',
                  fill=True, ax=ax, cbar=True)

belgium_boundary.plot(ax=ax, facecolor='none', edgecolor='black')
ax.set_title('Heatmap of AED Density')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()

### 5.2 Create a GeoDataframe from AED data

In [ ]:
# Convert to GeoDataFrame
gdf_aed_locations = gpd.GeoDataFrame(
    aed_locations_without_dups,
    geometry=gpd.points_from_xy(aed_locations_without_dups.Longitude, aed_locations_without_dups.Latitude)
)
print(len(gdf_aed_locations))

# Filter out invalid geometries
gdf_aed_locations = gdf_aed_locations[gdf_aed_locations['geometry'].is_valid]
print(len(gdf_aed_locations)) # No invalid geometries found

### 5.3 Adding AED Density to Dataset

#### 5.3.1 Count number of AEDs per grid

In [ ]:
# Spatial join the grid with the aed locations points
joined_aed = gpd.sjoin(gdf_aed_locations, grid, how='left')

# Count incidents in each grid cell
aed_counts = joined_aed.groupby('index_right').size()

# Map incidents counts to the grid
grid['aed_count'] = 0
grid.loc[aed_counts.index, 'aed_count'] = aed_counts.values

#### 5.3.2 Visualization of self-counted AED Density

In [ ]:
# Plot the AED density
fig, ax = plt.subplots(figsize=(12, 8))
grid.plot(column='aed_count', ax=ax, cmap='OrRd', alpha=0.5, legend=True,
         legend_kwds={"label": "Number of AEDs per Grid", "orientation": "horizontal"})
belgium_boundary.plot(ax=ax, facecolor='none', edgecolor='black')
ax.set_title('AED Density Grid')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()

#### 5.3.3 Combine incident and AED count to interventions data

In [ ]:
# Perform a spatial join to add the incident_count to the interventions data
gdf_interventions_with_both_count = gpd.sjoin(gdf_interventions_with_incident_count, grid[['geometry', 'aed_count']],
                                              how='left')
print(len(gdf_interventions_with_both_count))
gdf_interventions_with_both_count.drop(columns='index_right').head(5)

#### 5.3.4 Save Interventions Data with Counts to Repository

In [ ]:
gdf_interventions_with_both_count.to_csv(output_gdf_interventions_path, index=False)

## 6. Response Time threshhold

In [ ]:
# Only check Response times longer then 10 minutes
# (According to literature waiting longer ten 10 minutes could be life-threatening)
ResponseTimeFilterd = gdf_interventions_with_both_count[gdf_interventions_with_both_count['T3-T0'] > 10]
ResponseTimeFilterd_sorted = ResponseTimeFilterd.sort_values(by='T3-T0', ascending=True)


ResponseTimeFilterd_sorted_amb = ResponseTimeFilterd_sorted[ResponseTimeFilterd_sorted['Vector type'] == 'Ambulance']
ResponseTimeFilterd_sorted_mug = ResponseTimeFilterd_sorted[ResponseTimeFilterd_sorted['Vector type'] == 'MUG']
ResponseTimeFilterd_sorted_pit = ResponseTimeFilterd_sorted[ResponseTimeFilterd_sorted['Vector type'] == 'PIT']

fig, ax = plt.subplots(3, 1, figsize=(8,24))

sns.scatterplot(
    data=ResponseTimeFilterd_sorted_amb,
    x='Longitude', y='Latitude',
    hue='T3-T0', palette='YlOrRd',
    legend='brief', ax=ax[0]).set(
    title='Ambulance Response Times by Location', xlabel= 'Longitude', ylabel='Latitude')
sns.scatterplot(
    data=ResponseTimeFilterd_sorted_mug,
    x='Longitude', y='Latitude',
    hue='T3-T0', palette='YlOrRd',
    legend='brief', ax=ax[1]).set(
    title='MUG Response Times by Location', xlabel= 'Longitude', ylabel='Latitude')
sns.scatterplot(
    data=ResponseTimeFilterd_sorted_pit,
    x='Longitude', y='Latitude',
    hue='T3-T0', palette='YlOrRd',
    legend='brief', ax=ax[2]).set(
    title='PIT Response Times by Location', xlabel= 'Longitude', ylabel='Latitude')
belgium_with_provinces_boundary.plot(ax=ax[0], facecolor='none', edgecolor='black')
belgium_with_provinces_boundary.plot(ax=ax[1], facecolor='none', edgecolor='black')
belgium_with_provinces_boundary.plot(ax=ax[2], facecolor='none', edgecolor='black')

## 7. Identifying Hotspots or High-Risk Areas

Note: These thresholds are chosen arbitrary.

We defined a high risk area as an area of 3km² in which:
* There are more than 5 cardiac arrests
* The response times of these incidents are larger than 8 minutes
* There are less then 5 AEDs.

Literature says (https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7892213/): 

'We found that a short Emergency Medical Service (EMS) response time was associated with a high rate of survival 
to hospital discharge after Out-of-Hospital Cardiac Arrest (OHCA). 
The optimal response time threshold for survival to hospital discharge was 6.2min. 
In the case of OHCA in public areas or with bystander CPR, the threshold was prolonged to 7.2min and 6.3min, 
respectively; and in the absence of a witness, the threshold was shortened to 4.2min.'

About the incident density it says:

'Incident density, or the frequency of events in a specific area, can impact ambulance response times and survival 
outcomes for out-of-hospital cardiac arrest (OHCA) patients.
High incident density areas may lead to longer response times due to increased demand on emergency medical services (EMS)
resources.'

In [ ]:
# Define high-risk based on response time, incident frequency and aed density
response_time_threshold = 8  # minutes
incident_density_threshold = 5  # more than five previous interventions on that location area
aed_density_threshold = 5  # less than 5 aeds in the 3 km³ grid for that location


# Identify high-risk areas
high_risk_areas = gdf_interventions_with_both_count.loc[
    (gdf_interventions_with_both_count['T3-T0'] > response_time_threshold) &
    (gdf_interventions_with_both_count['incident_count'] > incident_density_threshold) &
    (gdf_interventions_with_both_count['aed_count'] < aed_density_threshold), :
    ]

# Reset indices
high_risk_areas.reset_index(drop=True, inplace = True)

print(
    'Number of high risk areas in Belgium according to chosen thresholds: ',
    len(high_risk_areas['geometry'].drop_duplicates()))


In [ ]:
# Visualise high-risk areas
fig, ax = plt.subplots(figsize=(12, 8))
belgium_with_provinces_boundary.plot(ax=ax, facecolor='none', edgecolor='black')
scatter = sns.scatterplot(data=high_risk_areas.sort_values(by='T3-T0', ascending=True),
                          x='Longitude', y='Latitude',
                          size='incident_count', hue='T3-T0', palette='coolwarm',
                          sizes=(20, 200), legend='auto', ax=ax)
ax.set_title('High-Risk Areas based on Response Time, Incident Frequency and AED Density')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()

## 8. Clustering High-Risk Areas Using DBSCAN as spatial clustering algorithm

The Density-Based Spatial Clustering of Applications with Noise (DBSCAN) algorithm views clusters as areas of high density separated by areas of low density. Due to this rather generic view, clusters found by DBSCAN can be any shape, as opposed to k-means which assumes that clusters are convex shaped.

To begin, DBSCAN has three hyperparameters:

1) Epsilon: two points are considered neighbors if they are closer than Epsilon.
2) minPts: Minimal neighbors for a point to be classified as a core point in a cluster. The minPts parameter is easy to set. The minPts should be 4 for a two-dimensional dataset. For multidimensional dataset, minPts should be 2 * number of dimensions.
3) The distance metric (Euclidean, Haversine..).

A core point has at least the minimal number of neighbors within a radius epsilon.


DBSCAN is not entirely deterministic: border points that are reachable from more than one cluster can be part of either cluster, depending on the order the data are processed. For most data sets and domains, this situation does not arise often and has little impact on the clustering result.

### 8.1 Determine hyperparameters for DBSCAN

In [ ]:
# Drop duplicates because same locations multiple times in dataset for representing response time of different vector types
df = high_risk_areas[['Latitude', 'Longitude']].drop_duplicates()

# Reset indices
df.reset_index(drop=True, inplace=True)

df

**minPts**

Should be 2 * number of dimensions. Here: 4, because two dimensions 'Latitude' and 'Longitude'.

In [ ]:
# min_samples should be 2 * number of dimensions
min_samples = 4

**Epsilon**

To determine the optimal ε parameter, the k-nearest neighbor (k-NN) distances are computed of an input dataset using the k-nearest neighbor method (unsupervised nearest neighbors learning). The NearestNeighbors function requires n_neighbors (number of neighbors) parameter, which can be same as the minPts value.

In the k-NN distance plot, you should look for the “knee” or “elbow” point (a threshold value where you see a sharp change) of the curve to find the optimal value of ε.

In [ ]:
# n_neighbors = 5 as k-neighbors function also returns distance of point to itself (i.e. first column will be zeros).
# So, only 4 usefull neighbors (columns) if 5 neighbors are chosen.
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(df)

# Find the k-neighbors of a point
distances, indices = neighbors_fit.kneighbors(df
                                             )
# sort the neighbor distances (lengths to points) in ascending order
distances_sorted = np.sort(distances, axis=0) # axis = 0 represents sort along first axis i.e. sort along row

k_dist = distances_sorted[:, 4]
plt.plot(k_dist)
plt.axhline(0.023, linestyle='--', color='red')
plt.ylabel("k-NN distance")
plt.xlabel("Sorted observations (4th NN)")
plt.show()

### 8.2 Compute DBSCAN clustering

This algorithm will consider a point as a core point if at least 5 points are within a distance of 3km

In [ ]:
# Initialize DBSCAN
dbscan = DBSCAN(eps=0.023, min_samples=min_samples)

# Perform clustering
df.loc[:, 'cluster'] = dbscan.fit_predict(df)

# Filter out noise points (cluster == -1)
df_filtered = df[df['cluster'] != -1]

### 8.3 Determine new AED locations (cluster centers)

In [ ]:
cluster_centers = df_filtered.groupby('cluster')[['Latitude', 'Longitude']].mean()

# Determine cluster sizes
cluster_sizes = df_filtered['cluster'].value_counts()
labels = dbscan.labels_

# Combine locations of clusters centers and their sizes
clusters_df = cluster_centers.merge(cluster_sizes, how='inner', on='cluster').rename(columns={'count': 'size'})

print('New AED Locations: \n')
clusters_df

In [ ]:
# Number of clusters in labels, ignoring noise if present.
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print("Estimated number of clusters: %d" % n_clusters)
print("Estimated number of noise points: %d" % n_noise)

# Evaluation metrics

sc = silhouette_score(df[['Latitude', 'Longitude']], labels)
print("Silhouette Coefficient:%0.2f" % sc)

### 8.3 Visualize new clusters and their size

In [ ]:
# Plot clusters
fig, ax = plt.subplots(figsize=(12, 8))
custom_palette = sns.color_palette("Paired", len(cluster_sizes) + 1)

# Plot the proposed AED locations
new_aed_gdf = gpd.GeoDataFrame(cluster_centers,
                               geometry=gpd.points_from_xy(cluster_centers['Longitude'], cluster_centers['Latitude']))
new_aed_gdf.plot(ax=ax, marker='x', color='blue', markersize=100, label='Proposed AED Locations')


# Overlay the plot the high-risk areas and color by cluster
scatter = sns.scatterplot(
    data=df,
    x='Longitude', y='Latitude',
    hue='cluster', palette=custom_palette,
    legend=False, ax=ax)
# Overlay the Belgium boundary
belgium_boundary.plot(ax=ax, facecolor='none', edgecolor='black')



ax.set_title('High-Risk Areas Clusters and Proposed AED Locations')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
# New plot for cluster sizes
fig, ax = plt.subplots(figsize=(12, 8))

# Plot only cluster centers with sizes and colors depending on cluster size to determine most 'problematic' clusters
belgium_with_provinces_boundary.plot(ax=ax, facecolor='none', edgecolor='black')
scatter = sns.scatterplot(data=clusters_df,
                          x='Longitude', y='Latitude', size='size', sizes=(20, 200),
                          hue='size', palette='coolwarm', legend='brief')
ax.set_title('Cluster Centers with Sizes Proportional to Cluster Sizes')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## 9. Save new AED locations to a new dataset

In [ ]:
# Save cluster centers to new csv file
clusters_df.to_csv(output_aed_path, index=True)